In [ ]:
from catboost import CatBoostRegressor
import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import *
from src.data_split import *
from src.metrics import *


In [ ]:
train_df = pd.read_csv('../data/train_test/holdout/train_full.csv', index_col='id', parse_dates=['timestamp'])
test_df = pd.read_csv('../data/train_test/holdout/holdout.csv', index_col='id', parse_dates=['timestamp'])

C:\Users\sokol\AppData\Local\Temp\ipykernel_13668\3531396417.py:1: DtypeWarning: Columns (0: old_education_build_share, 1: provision_doctors) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv('../data/train_test/holdout/train_full.csv', index_col='id', parse_dates=['timestamp'])
C:\Users\sokol\AppData\Local\Temp\ipykernel_13668\3531396417.py:2: DtypeWarning: Columns (0: load_of_teachers_school_per_teacher) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv('../data/train_test/holdout/holdout.csv', index_col='id', parse_dates=['timestamp'])


In [ ]:
time_folds = get_time_folds()

In [ ]:
cv_results = []
models = []
for fold in time_folds:
    train, val = split_fold(df=train_df, fold=fold)
    train, val = base_preprocessing(train), base_preprocessing(val)

    X_train = train.drop('price_doc', axis=1)
    X_val = val.drop('price_doc', axis=1)

    y_train = train['price_doc']
    y_val = val['price_doc']

    cat_features = X_train.select_dtypes(include='category').columns

    model = CatBoostRegressor(
        iterations=3000,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        early_stopping_rounds=200,
        verbose=200,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_val, y_val),
        use_best_model=True,
    )

    pred_log = model.predict(X_val)
    pred = np.expm1(pred_log)

    score = rmsle(y_val, pred)

    cv_results.append({
        "fold": fold["fold"],
        "train_blocks": ",".join(fold["train_blocks"]),
        "val_block": fold["val_block"],
        "rmsle": score,
        "best_iteration": model.get_best_iteration(),
    })

    models.append(model)

cv_results = pd.DataFrame(cv_results)

cv_results

    

CatBoostError: feature names should be a sequence, but got Index(['timestamp', 'product_type', 'sub_area', 'culture_objects_top_25',
       'thermal_power_plant_raion', 'incineration_raion',
       'oil_chemistry_raion', 'radiation_raion', 'railroad_terminal_raion',
       'big_market_raion', 'nuclear_reactor_raion', 'detention_facility_raion',
       'water_1line', 'big_road1_1line', 'railroad_1line', 'ecology',
       'time_block'],
      dtype='str')

<class 'pandas.DataFrame'>
Index: 10160 entries, 0 to 10164
Columns: 391 entries, timestamp to apartment_fund_sqm
dtypes: category(17), float64(219), int64(155)
memory usage: 29.3 MB
